# Step 09 — Diagnostics & Relative Rotation Graph

This notebook explores ratio diagnostics and Relative Rotation Graph (RRG)
analysis from pipeline step 8 (`pipelines/08_diagnostics.py`).

**RRG concept**: each asset is placed on a 2D plane with:
- **RS-Ratio** (x-axis): relative strength vs benchmark (>100 = outperforming)
- **RS-Momentum** (y-axis): rate of change of RS (>100 = improving)

Four quadrants: LEADING (strong+accelerating), WEAKENING (strong+decelerating),
LAGGING (weak+decelerating), IMPROVING (weak+accelerating). Assets typically
rotate clockwise through quadrants.

**Run `python pipelines/08_diagnostics.py` before executing this notebook.**

## Setup & Load Data (D7.1)

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from trading_crab_lib.config import load, setup_logging
from trading_crab_lib.runtime import RunConfig
from trading_crab_lib import DATA_DIR, OUTPUT_DIR, plotting
setup_logging("INFO")
log = logging.getLogger("09_diagnostics")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=False)

DIAG_DIR = OUTPUT_DIR / "reports" / "diagnostics"
RAW_DIR = DATA_DIR / "raw"

In [ ]:
import subprocess
from pathlib import Path

def run_step_if_needed(step: int, required_paths: list, auto_run: bool = True) -> bool:
    """Run the pipeline step if any required output files are missing."""
    missing = [p for p in required_paths if not Path(p).exists()]
    if not missing:
        return True
    print(f"Missing: {[str(p) for p in missing]}")
    scripts = sorted(Path("../pipelines").glob(f"{step:02d}_*.py"))
    if not scripts:
        print(f"No pipeline script found for step {step}.")
        return False
    script = scripts[0]
    if not auto_run:
        print(f"  → Run: python {script}")
        return False
    print(f"  → Running {script.name} ...")
    result = subprocess.run(["python", str(script)], capture_output=True, text=True, cwd="..")
    out = result.stdout
    if len(out) > 4000:
        out = out[:2000] + "\n...\n" + out[-2000:]
    print(out)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
        return False
    print(f"  ✓ Step {step} complete.")
    return True

run_step_if_needed(8, [DIAG_DIR / "rrg_current.parquet"], auto_run=False)

In [ ]:
rrg_df = None
ratios_df = None
prices = None

# RRG data
rrg_path = DIAG_DIR / "rrg_current.parquet"
try:
    rrg_df = pd.read_parquet(rrg_path)
    print(f"RRG data loaded: {rrg_df.shape}")
    print(f"  Columns: {list(rrg_df.columns)}")
    if 'asset' in rrg_df.columns:
        print(f"  Assets: {sorted(rrg_df['asset'].unique())}")
except FileNotFoundError:
    print(f"RRG data not found at {rrg_path}")
    print("Run: python pipelines/08_diagnostics.py")

# Ratio diagnostics
ratios_path = DIAG_DIR / "ratios.parquet"
try:
    ratios_df = pd.read_parquet(ratios_path)
    print(f"Ratio diagnostics loaded: {ratios_df.shape}")
except FileNotFoundError:
    print("Ratio diagnostics not found — will compute from prices if available.")

# Asset prices (for computing diagnostics on the fly)
prices_path = RAW_DIR / "asset_prices.parquet"
try:
    prices = pd.read_parquet(prices_path)
    print(f"Asset prices loaded: {prices.shape}")
except FileNotFoundError:
    print("Asset prices not found — some cells will be skipped.")

## RRG 4-Quadrant Scatter Plot (D7.2)

Each dot is an asset positioned by its RS-Ratio (x) and RS-Momentum (y).
Assets in LEADING are outperforming and accelerating; LAGGING assets are
underperforming and decelerating. The typical clockwise rotation
(Improving → Leading → Weakening → Lagging) reflects the asset lifecycle.

In [ ]:
if rrg_df is not None and not rrg_df.empty:
    # Adapt column names: rrg_for_benchmark uses rs_ratio/rs_momentum,
    # plot_rrg_scatter expects rs/rm
    plot_rrg = rrg_df.copy()
    col_map = {"rs_ratio": "rs", "rs_momentum": "rm"}
    plot_rrg = plot_rrg.rename(columns=col_map)

    if "asset" in plot_rrg.columns:
        plot_rrg = plot_rrg.set_index("asset")

    plotting.plot_rrg_scatter(plot_rrg, run_cfg)

    # Print quadrant summary
    quad_col = "quadrant" if "quadrant" in plot_rrg.columns else None
    if quad_col:
        print("\nQuadrant Summary:")
        for q in ["LEADING", "WEAKENING", "LAGGING", "IMPROVING"]:
            assets = plot_rrg[plot_rrg[quad_col] == q].index.tolist()
            print(f"  {q:12s}: {', '.join(assets) if assets else '(none)'}")
elif prices is not None and "SPY" in prices.columns:
    # Compute RRG on the fly
    from trading_crab_lib.diagnostics import rrg_for_benchmark
    rrg_df = rrg_for_benchmark(prices, "SPY")
    if not rrg_df.empty:
        plot_rrg = rrg_df.rename(columns={"rs_ratio": "rs", "rs_momentum": "rm"})
        if "asset" in plot_rrg.columns:
            plot_rrg = plot_rrg.set_index("asset")
        plotting.plot_rrg_scatter(plot_rrg, run_cfg)
    else:
        print("RRG computation returned empty — insufficient price data.")
else:
    print("No RRG data or prices available.")

## Rolling Z-Score Time-Series (D7.3)

Rolling z-scores for 4 key price ratios from the diagnostics config.
Values beyond +/-2σ signal extreme divergence from the trailing norm —
potential regime change or mean-reversion opportunity.

In [ ]:
if prices is not None:
    from trading_crab_lib.diagnostics import rolling_zscore

    ratios_cfg = cfg.get("diagnostics", {}).get("ratios") or []
    if not ratios_cfg:
        # Default ratios if config is empty
        ratios_cfg = [
            {"name": "SPY/TLT", "numerator": "SPY", "denominator": "TLT"},
            {"name": "GLD/TLT", "numerator": "GLD", "denominator": "TLT"},
            {"name": "SPY/GLD", "numerator": "SPY", "denominator": "GLD"},
            {"name": "USO/GLD", "numerator": "USO", "denominator": "GLD"},
        ]

    n_ratios = len(ratios_cfg)
    fig, axes = plt.subplots(n_ratios, 1, figsize=(14, 3 * n_ratios), sharex=True, squeeze=False)

    for idx, item in enumerate(ratios_cfg):
        ax = axes[idx][0]
        num = item.get("numerator", "")
        den = item.get("denominator", "")
        name = item.get("name", f"{num}/{den}")

        if num not in prices.columns or den not in prices.columns:
            ax.set_title(f"{name} — data unavailable", fontsize=10)
            continue

        ratio = prices[num] / prices[den].replace(0, np.nan)
        ratio = ratio.dropna()
        z = rolling_zscore(ratio, window=20)

        ax.plot(z.index, z.values, color=plotting.CUSTOM_COLORS[0], linewidth=1)
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.axhline(2, color="red", linewidth=0.5, linestyle="--", alpha=0.5)
        ax.axhline(-2, color="red", linewidth=0.5, linestyle="--", alpha=0.5)
        ax.fill_between(z.index, 2, z.values, where=z.values > 2,
                        color="red", alpha=0.15)
        ax.fill_between(z.index, -2, z.values, where=z.values < -2,
                        color="blue", alpha=0.15)
        ax.set_ylabel("z-score")
        ax.set_title(f"{name} — Rolling Z-Score (window=20)", fontsize=10)
        ax.grid(alpha=0.2)

    axes[-1][0].set_xlabel("Date")
    fig.suptitle("Ratio Z-Scores — Extreme Values Signal Divergence", fontsize=13, y=1.01)
    fig.tight_layout()
    plotting._save_or_show(fig, "09_ratio_zscores.png", run_cfg)
else:
    print("Asset prices not available.")

## Quadrant Rotation History (D7.4)

For each asset, compute RRG quadrant at each quarter and show how
often it resides in each quadrant. Assets with high LEADING frequency
are consistent outperformers; high LAGGING frequency indicates
persistent underperformance.

In [ ]:
if prices is not None and "SPY" in prices.columns:
    from trading_crab_lib.diagnostics import normalize_100
    import seaborn as sns

    benchmark = "SPY"
    rs_window = 12
    rm_window = 4
    quarterly = prices.resample("QE").last()

    tickers = [t for t in quarterly.columns if t != benchmark and quarterly[t].notna().sum() > rs_window + rm_window]

    # Compute quadrant for each asset at each quarter
    quad_history = {}
    for ticker in tickers:
        asset_p = quarterly[ticker].dropna()
        bench_p = quarterly[benchmark].reindex(asset_p.index).dropna()
        common = asset_p.index.intersection(bench_p.index)
        if len(common) < rs_window + rm_window + 1:
            continue
        cum_a = (asset_p.loc[common].pct_change().fillna(0) + 1).cumprod()
        cum_b = (bench_p.loc[common].pct_change().fillna(0) + 1).cumprod()
        rs = normalize_100(cum_a / cum_b, center_window=rs_window)
        rm = normalize_100(rs.diff().dropna(), center_window=rm_window)
        common2 = rs.index.intersection(rm.index)
        quads = []
        for dt in common2:
            r, m = rs.loc[dt], rm.loc[dt]
            if pd.isna(r) or pd.isna(m):
                continue
            if r > 100 and m > 100:
                quads.append("LEADING")
            elif r > 100:
                quads.append("WEAKENING")
            elif m > 100:
                quads.append("IMPROVING")
            else:
                quads.append("LAGGING")
        if quads:
            quad_history[ticker] = pd.Series(quads).value_counts(normalize=True)

    if quad_history:
        quad_freq = pd.DataFrame(quad_history).T.fillna(0)
        # Ensure column order
        for q in ["LEADING", "IMPROVING", "WEAKENING", "LAGGING"]:
            if q not in quad_freq.columns:
                quad_freq[q] = 0.0
        quad_freq = quad_freq[["LEADING", "IMPROVING", "WEAKENING", "LAGGING"]]
        quad_freq = quad_freq.sort_values("LEADING", ascending=True)

        quad_colors = {"LEADING": "#50a000", "IMPROVING": "#0000d0",
                       "WEAKENING": "#f48c06", "LAGGING": "#d00000"}

        fig, ax = plt.subplots(figsize=(10, max(4, len(quad_freq) * 0.35)))
        left = np.zeros(len(quad_freq))
        for q in ["LEADING", "IMPROVING", "WEAKENING", "LAGGING"]:
            ax.barh(quad_freq.index, quad_freq[q], left=left,
                    color=quad_colors[q], label=q, alpha=0.85)
            left += quad_freq[q].values
        ax.set_xlabel("Fraction of Quarters")
        ax.set_title(f"Quadrant Frequency per Asset (vs {benchmark})", fontsize=12)
        ax.legend(loc="lower right", fontsize=8)
        ax.set_xlim(0, 1)
        ax.tick_params(labelsize=8)
        fig.tight_layout()
        plotting._save_or_show(fig, "09_quadrant_rotation_history.png", run_cfg)
    else:
        print("Insufficient data to compute quadrant history.")
else:
    print("Asset prices or SPY benchmark not available.")

## Ratio Percentile Rank Dashboard (D7.5)

Current value of each ratio positioned within its historical distribution.
Shows whether the current reading is historically high (>80th percentile),
low (<20th), or normal. Extreme percentile ranks often precede reversals.

In [ ]:
if prices is not None:
    from trading_crab_lib.diagnostics import percentile_rank

    ratios_cfg = cfg.get("diagnostics", {}).get("ratios") or []
    if not ratios_cfg:
        ratios_cfg = [
            {"name": "SPY/TLT", "numerator": "SPY", "denominator": "TLT"},
            {"name": "GLD/TLT", "numerator": "GLD", "denominator": "TLT"},
            {"name": "SPY/GLD", "numerator": "SPY", "denominator": "GLD"},
            {"name": "USO/GLD", "numerator": "USO", "denominator": "GLD"},
        ]

    rows = []
    fig, axes = plt.subplots(len(ratios_cfg), 1, figsize=(12, 2.5 * len(ratios_cfg)),
                             squeeze=False)

    for idx, item in enumerate(ratios_cfg):
        ax = axes[idx][0]
        num = item.get("numerator", "")
        den = item.get("denominator", "")
        name = item.get("name", f"{num}/{den}")

        if num not in prices.columns or den not in prices.columns:
            ax.set_title(f"{name} — data unavailable")
            continue

        ratio = (prices[num] / prices[den].replace(0, np.nan)).dropna()
        pct = percentile_rank(ratio, window=len(ratio))  # full-history rank

        current_val = ratio.iloc[-1] if len(ratio) > 0 else np.nan
        current_pct = pct.iloc[-1] if len(pct) > 0 else np.nan

        # Histogram of ratio values with current value marked
        ax.hist(ratio.values, bins=40, color=plotting.CUSTOM_COLORS[0], alpha=0.6,
                edgecolor="white")
        if not np.isnan(current_val):
            ax.axvline(current_val, color=plotting.CUSTOM_COLORS[1], linewidth=2,
                       linestyle="-", label=f"Current: {current_val:.3f} ({current_pct:.0%} rank)")
        ax.set_title(f"{name}", fontsize=10)
        ax.legend(fontsize=8)
        ax.tick_params(labelsize=7)

        rows.append({"Ratio": name, "Current": f"{current_val:.4f}",
                     "Percentile": f"{current_pct:.0%}",
                     "Signal": "HIGH" if current_pct > 0.8 else "LOW" if current_pct < 0.2 else "NORMAL"})

    fig.suptitle("Current Ratio Values vs Historical Distribution", fontsize=13, y=1.01)
    fig.tight_layout()
    plotting._save_or_show(fig, "09_percentile_rank_dashboard.png", run_cfg)

    # Summary table
    if rows:
        print("\nPercentile Rank Summary:")
        display(pd.DataFrame(rows).set_index("Ratio"))
else:
    print("Asset prices not available.")